# 長い ALS の収束（T2）

設計：`plan/50_llm_tt_vs_matrix/10_qwen3_0p6b_base/03_long_als_convergence/design.md`

この Notebook は予備実験（`01_single_layer_pilot`）とベンチマーク（`02_metric_tt_solver_benchmark`）のファイルを**読むだけ**である。書き込みは `03_long_als_convergence` の models と results だけにする。

本実行は `RUN_FULL = False` のあいだは走らない。試運転は `RUN_SMOKE = True` のときだけ走る。


## 1. 設定

- 入力: プロジェクトルート。設計 §3 と §5 の固定値。
- このステップで行うこと: フラグ、ソルバー設定、停止の規則、出力先、GPU と torch の版を決める。`RUN_SMOKE` のときは `settings_smoke.json` を書く。
- 行わないこと: 重みの読み込み、ALS、予備実験のファイルへの書き込み、設定値の変更。
- 出力: `RUN_SMOKE`、`RUN_FULL`、`CONFIG`、`FULL_RULE`、`SMOKE_RULE`、`EQUIVALENCE_TOLERANCE`、`RESULTS`、`MODELS`。
- 完了条件: `CONFIG` が float32_refine・Jacobi・fixed・1e-8。`FULL_RULE` は既定のまま。`SMOKE_RULE` は 5 掃引、2 掃引ごとに保存。
- 次のステップへの接続: 2 は `PILOT_DATA` と `RESULTS` を使う。


In [2]:
from dataclasses import asdict
from pathlib import Path
import time

import torch

from nn_compression.tt_mps.metric_tt_solver import ConvergenceStopRule, MetricTTSolverConfig
from nn_compression.utils.artifacts import atomic_write_json
from nn_compression.utils.paths import find_project_root, get_experiment_dirs

RUN_SMOKE = False
RUN_FULL = True
EQUIVALENCE_TOLERANCE = 1e-3

CONFIG = MetricTTSolverConfig(
    device="cuda",
    precision="float32_refine",
    preconditioner="jacobi",
    cg_tolerance_mode="fixed",
    cg_relative_tolerance=1e-8,
    max_cg_iterations=500,
    max_refinements=5,
)
FULL_RULE = ConvergenceStopRule()
SMOKE_RULE = ConvergenceStopRule(max_sweeps=5, checkpoint_every=2)

PROJECT = find_project_root(Path.cwd())
_pilot_data, PILOT_MODELS, PILOT_RESULTS = get_experiment_dirs(
    PROJECT, "50_llm_tt_vs_matrix", "10_qwen3_0p6b_base", "01_single_layer_pilot", create=False,
)
PILOT_DATA = _pilot_data / "50_llm_tt_vs_matrix" / "10_qwen3_0p6b_base" / "01_single_layer_pilot"
_bench_data, _bench_models, BENCH_RESULTS = get_experiment_dirs(
    PROJECT, "50_llm_tt_vs_matrix", "10_qwen3_0p6b_base", "02_metric_tt_solver_benchmark", create=False,
)
_data, MODELS, RESULTS = get_experiment_dirs(
    PROJECT, "50_llm_tt_vs_matrix", "10_qwen3_0p6b_base", "03_long_als_convergence", create=False,
)
MODELS.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA が使えません。")

ENVIRONMENT = {
    "gpu_name": torch.cuda.get_device_name(0),
    "torch_version": torch.__version__,
    "cuda_version": torch.version.cuda,
}
SETTINGS = {
    "RUN_SMOKE": RUN_SMOKE,
    "RUN_FULL": RUN_FULL,
    "equivalence_tolerance": EQUIVALENCE_TOLERANCE,
    "solver_config": {**asdict(CONFIG), "device": str(CONFIG.device)},
    "full_rule": asdict(FULL_RULE),
    "smoke_rule": asdict(SMOKE_RULE),
    "environment": ENVIRONMENT,
}
print(ENVIRONMENT, flush=True)
print("FULL_RULE", asdict(FULL_RULE), flush=True)
print("SMOKE_RULE", asdict(SMOKE_RULE), flush=True)


{'gpu_name': 'NVIDIA GeForce RTX 5070 Ti', 'torch_version': '2.11.0+cu128', 'cuda_version': '12.8'}
FULL_RULE {'relative_decrease_tolerance': 1e-07, 'patience_sweeps': 5, 'stationarity_tolerance': 1e-06, 'max_sweeps': 1000, 'local_residual_tolerance': 1e-08, 'retry_max_refinements': 10, 'monotone_tolerance': 1e-10, 'checkpoint_every': 50}
SMOKE_RULE {'relative_decrease_tolerance': 1e-07, 'patience_sweeps': 5, 'stationarity_tolerance': 1e-06, 'max_sweeps': 5, 'local_residual_tolerance': 1e-08, 'retry_max_refinements': 10, 'monotone_tolerance': 1e-10, 'checkpoint_every': 2}


## 2. 入力

- 入力: 予備実験の data・models・results と、ベンチマークの `als_trajectory.csv`。読むだけ。
- このステップで行うこと: W、W_gate、較正の G と G_λ、検証の Gram と活性化、比較用 CSV を読み、各ファイルの SHA-256 を記録する。`per_document` と `calibration_activations.pt` は使わない。
- 行わないこと: 正準化、M4_retruncated の作成、ALS。
- 出力: `WEIGHT`、`W_GATE`、`G`、`G_LAMBDA`、`VALIDATION`、`REFERENCE`、`INPUT_SHA256`。`settings_smoke.json`（試運転のとき）。
- 完了条件: W と W_gate が `(3072, 1024)` の float64。G_λ が `(1024, 1024)`。検証の活性化は文書 × 1024 × 1024。
- 次のステップへの接続: 3 は M3・M4 のファイルと `REFERENCE["initial"]` を使う。


In [4]:
import csv

from nn_compression.compression.metric_low_rank import regularize_gram
from nn_compression.utils.artifacts import file_sha256

INPUT_PATHS = {
    "weights_float64.pt": PILOT_DATA / "weights_float64.pt",
    "calibration_grams.pt": PILOT_DATA / "calibration_grams.pt",
    "validation_same_grams.pt": PILOT_DATA / "validation_same_grams.pt",
    "validation_other_grams.pt": PILOT_DATA / "validation_other_grams.pt",
    "validation_same_activations.pt": PILOT_DATA / "validation_same_activations.pt",
    "validation_other_activations.pt": PILOT_DATA / "validation_other_activations.pt",
    "m3_T2_50.pt": PILOT_MODELS / "m3_T2_50.pt",
    "m3_T2_25.pt": PILOT_MODELS / "m3_T2_25.pt",
    "m4_T2_50.pt": PILOT_MODELS / "m4_T2_50.pt",
    "m4_T2_25.pt": PILOT_MODELS / "m4_T2_25.pt",
    "m5_initial_values.csv": PILOT_RESULTS / "m5_initial_values.csv",
    "m5_history.csv": PILOT_RESULTS / "m5_history.csv",
    "rho.csv": PILOT_RESULTS / "rho.csv",
    "als_trajectory.csv": BENCH_RESULTS / "als_trajectory.csv",
}
missing = [name for name, path in INPUT_PATHS.items() if not path.is_file()]
if missing:
    raise FileNotFoundError(f"入力がありません: {missing}")
INPUT_SHA256 = {name: file_sha256(path) for name, path in INPUT_PATHS.items()}

weights = torch.load(INPUT_PATHS["weights_float64.pt"], map_location="cpu", weights_only=False)
WEIGHT = weights["W"]
W_GATE = weights["W_gate"]
if WEIGHT.shape != (3072, 1024) or W_GATE.shape != WEIGHT.shape:
    raise RuntimeError(f"重みの shape が違います: {tuple(WEIGHT.shape)} {tuple(W_GATE.shape)}")
if WEIGHT.dtype != torch.float64 or WEIGHT.device.type != "cpu":
    raise RuntimeError("W は float64 CPU である必要があります。")

calibration = torch.load(INPUT_PATHS["calibration_grams.pt"], map_location="cpu", weights_only=False)
G = calibration["total"]
del calibration
G_LAMBDA = regularize_gram(G, 1e-6)
if G.shape != (1024, 1024) or G_LAMBDA.shape != (1024, 1024):
    raise RuntimeError(f"Gram の shape が違います: {tuple(G.shape)}")

VALIDATION = {}
for domain, gram_name, activation_name in (
    ("validation_same", "validation_same_grams.pt", "validation_same_activations.pt"),
    ("validation_other", "validation_other_grams.pt", "validation_other_activations.pt"),
):
    gram_payload = torch.load(INPUT_PATHS[gram_name], map_location="cpu", weights_only=False)
    activation_payload = torch.load(INPUT_PATHS[activation_name], map_location="cpu", weights_only=False)
    activation = activation_payload["activation"]
    if activation.ndim != 3 or activation.shape[1:] != (1024, 1024):
        raise RuntimeError(f"{domain} の活性化 shape が {tuple(activation.shape)} です。")
    VALIDATION[domain] = {"gram": gram_payload["total"], "activation": activation}
    print(domain, "documents", int(activation.shape[0]), flush=True)
    del gram_payload, activation_payload


def read_csv(path):
    with path.open(encoding="utf-8", newline="") as handle:
        return list(csv.DictReader(handle))


REFERENCE = {
    "initial": read_csv(INPUT_PATHS["m5_initial_values.csv"]),
    "history": read_csv(INPUT_PATHS["m5_history.csv"]),
    "rho": read_csv(INPUT_PATHS["rho.csv"]),
    "als": read_csv(INPUT_PATHS["als_trajectory.csv"]),
}
SETTINGS["input_sha256"] = INPUT_SHA256
if RUN_SMOKE:
    atomic_write_json(RESULTS / "settings_smoke.json", SETTINGS)
print("inputs", len(INPUT_SHA256), flush=True)


validation_same documents 32
validation_other documents 32
inputs 14


## 3. 初期値

- 入力: `m3_T2_{50,25}.pt`、`m4_T2_{50,25}.pt`、`REFERENCE["initial"]`。
- このステップで行うこと: T2 の 50%・25% について、M3 は保存されたコアをそのまま使う。M4_retruncated は `DiagonalMetricTT` を作り、`prepare_m5_from_diagonal_tt` に M3 の rank を渡して作り直す。mode は M3 のファイルから取る。CPU の `q_lambda_of_cores` を予備実験の `q_lambda_before_als` と比べる。
- 行わないこと: M3 の正準化、ALS、検証データでの誤差。
- 出力: `JOBS`。各要素は `budget`、`initial`、`cores`、`out_modes`、`in_modes`、`expected_q`。
- 完了条件: 4 本ある。CPU で計算した初期 Q_λ が CSV と相対 1e-10 以内。公式の検算 1 は、試運転でソルバーが返す `initial_q_lambda` に対してもう一度行う。
- 次のステップへの接続: 4 の診断と 5 の試運転は `JOBS` を渡す。


In [6]:
from nn_compression.tt_mps.matrix import tt_matrix_ranks
from nn_compression.tt_mps.metric_tt import (
    DiagonalMetricTT,
    prepare_m5_from_diagonal_tt,
    q_lambda_of_cores,
)


def relative_gap(got, expected):
    return abs(float(got) - float(expected)) / abs(float(expected))


def expected_initial_q(budget, initial):
    matches = [
        row for row in REFERENCE["initial"]
        if row["calibration_set"] == "full"
        and row["tensorization"] == "T2"
        and row["budget"] == budget
        and row["initial"] == initial
    ]
    if len(matches) != 1:
        raise RuntimeError(f"初期値の参照が {len(matches)} 行です: {budget} {initial}")
    return float(matches[0]["q_lambda_before_als"])


JOBS = []
for budget in ("50", "25"):
    m3_payload = torch.load(PILOT_MODELS / f"m3_T2_{budget}.pt", map_location="cpu", weights_only=False)
    out_modes = tuple(int(mode) for mode in m3_payload["out_modes"])
    in_modes = tuple(int(mode) for mode in m3_payload["in_modes"])
    m3_cores = [core.detach().to(dtype=torch.float64, device="cpu").contiguous() for core in m3_payload["cores"]]
    m4_payload = torch.load(PILOT_MODELS / f"m4_T2_{budget}.pt", map_location="cpu", weights_only=False)
    fitted = DiagonalMetricTT(
        scaled_cores=m4_payload["scaled_cores"],
        input_scale_inv=m4_payload["input_scale_inv"],
        absorbed_cores=m4_payload["absorbed_cores"],
        scales=m4_payload["scales"],
        max_rank=m4_payload["max_rank"],
    )
    m4_cores = prepare_m5_from_diagonal_tt(fitted, out_modes, in_modes, tt_matrix_ranks(m3_cores))
    prepared = {"M3": m3_cores, "M4_retruncated": m4_cores}
    for initial, cores in prepared.items():
        expected = expected_initial_q(budget, initial)
        got = float(q_lambda_of_cores(cores, WEIGHT, G_LAMBDA, out_modes, in_modes).item())
        gap = relative_gap(got, expected)
        print(budget, initial, "cpu_q", f"{got:.8e}", "csv", f"{expected:.8e}", "rel", f"{gap:.3e}", flush=True)
        if gap > 1e-10:
            raise RuntimeError(f"検算 1（CPU）が通りません: {budget} {initial} 相対差 {gap}")
        JOBS.append({
            "budget": budget,
            "initial": initial,
            "cores": cores,
            "out_modes": out_modes,
            "in_modes": in_modes,
            "expected_q": expected,
        })
if len(JOBS) != 4:
    raise RuntimeError(f"初期値が {len(JOBS)} 本です。")


50 M3 cpu_q 1.20980141e+08 csv 1.20980141e+08 rel 0.000e+00
50 M4_retruncated cpu_q 2.36066768e+08 csv 2.36066768e+08 rel 1.262e-16
25 M3 cpu_q 1.60255804e+08 csv 1.60255804e+08 rel 1.860e-16
25 M4_retruncated cpu_q 2.75675344e+08 csv 2.75675344e+08 rel 1.946e-15


## 4. 診断

- 入力: `WEIGHT`、`W_GATE`、`G`、`VALIDATION`。
- このステップで行うこと: 保存のときに呼ぶ診断関数と、掃引記録を CSV の行にする関数を定義する。診断は停止の判定に使わない。
- 行わないこと: ALS の実行。
- 出力: `make_diagnostics`、`history_row`、`diagnostics_row`、`job_metadata`、`checkpoint_path`。
- 完了条件: 診断の戻り値のキーが `e_validation_same`、`e_validation_other`、`E_gated_validation_same`、`E_gated_validation_other`、`e_calibration`。
- 次のステップへの接続: 5 と 6 がこの関数をソルバーに渡す。


In [8]:
from nn_compression.compression.metric_low_rank import (
    gated_error_parts_per_document,
    output_error_ratios_from_gram,
    squared_output_error_from_gram,
)
from nn_compression.tt_mps.matrix import tt_matrix_to_dense

DIAGNOSTIC_KEYS = (
    "e_validation_same",
    "e_validation_other",
    "E_gated_validation_same",
    "E_gated_validation_other",
    "e_calibration",
)
TIMING_FIELDS = {
    "seconds",
    "seconds_hvp",
    "seconds_preconditioner",
    "seconds_preconditioner_apply",
    "seconds_qr",
    "seconds_accept",
    "seconds_stationarity",
    "seconds_q_unregularized",
    "peak_memory_bytes",
}
HISTORY_FIELDS = (
    "sweep",
    "q_lambda",
    "q_unregularized",
    "relative_decrease",
    "small_decrease_streak",
    "monotone_ok",
    "s_0",
    "s_1",
    "s_2",
    "stationarity_max",
    "update_sites",
    "cg_iterations",
    "cg_residuals",
    "refinements",
    "retried",
    "accepted",
    "n_rejected",
    "seconds",
    "seconds_hvp",
    "seconds_preconditioner",
    "seconds_preconditioner_apply",
    "seconds_qr",
    "seconds_accept",
    "seconds_stationarity",
    "seconds_q_unregularized",
    "peak_memory_bytes",
    "stop_reason",
)


def diagnostics_for_job(job):
    out_modes = job["out_modes"]
    in_modes = job["in_modes"]

    def diagnostics(cores, sweep):
        del sweep
        approximated = tt_matrix_to_dense(cores, out_modes, in_modes)
        values = {"e_calibration": output_error_ratios_from_gram(WEIGHT, approximated, G)["e"]}
        for domain in ("validation_same", "validation_other"):
            bundle = VALIDATION[domain]
            values[f"e_{domain}"] = output_error_ratios_from_gram(WEIGHT, approximated, bundle["gram"])["e"]
            numerator, denominator = gated_error_parts_per_document(
                WEIGHT, approximated, W_GATE, list(bundle["activation"].unbind(0)),
            )
            values[f"E_gated_{domain}"] = float((numerator.sum() / denominator.sum()).item())
        return values
    return diagnostics


def join_values(values):
    return ";".join(str(item) for item in values)


def history_row(budget, initial, record):
    if len(record.stationarity) != 3:
        raise RuntimeError(f"s_k の長さが {len(record.stationarity)} です。T2 は 3 です。")
    row = {
        "budget": budget,
        "initial": initial,
        "sweep": record.sweep,
        "q_lambda": record.q_lambda,
        "q_unregularized": record.q_unregularized,
        "relative_decrease": record.relative_decrease,
        "small_decrease_streak": record.small_decrease_streak,
        "monotone_ok": record.monotone_ok,
        "s_0": record.stationarity[0],
        "s_1": record.stationarity[1],
        "s_2": record.stationarity[2],
        "stationarity_max": record.stationarity_max,
        "update_sites": join_values(record.update_sites),
        "cg_iterations": join_values(record.cg_iterations),
        "cg_residuals": join_values(record.cg_residuals),
        "refinements": join_values(record.refinements),
        "retried": join_values(record.retried),
        "accepted": join_values(record.accepted),
        "n_rejected": record.n_rejected,
        "seconds": record.seconds,
        "seconds_hvp": record.seconds_hvp,
        "seconds_preconditioner": record.seconds_preconditioner,
        "seconds_preconditioner_apply": record.seconds_preconditioner_apply,
        "seconds_qr": record.seconds_qr,
        "seconds_accept": record.seconds_accept,
        "seconds_stationarity": record.seconds_stationarity,
        "seconds_q_unregularized": record.seconds_q_unregularized,
        "peak_memory_bytes": record.peak_memory_bytes,
        "stop_reason": record.stop_reason or "",
    }
    return row


def diagnostics_row(budget, initial, values, timing):
    row = {"budget": budget, "initial": initial, "sweep": values["sweep"]}
    for key in DIAGNOSTIC_KEYS:
        row[key] = values[key]
    row["seconds_diagnostics"] = timing["seconds_diagnostics"]
    row["seconds_save"] = timing["seconds_save"]
    return row


def job_metadata(budget, initial, run_kind):
    return {
        "budget": budget,
        "initial": initial,
        "run_kind": run_kind,
        "input_sha256": dict(INPUT_SHA256),
    }


def checkpoint_path(budget, initial, suffix):
    return MODELS / f"long_als_T2_{budget}_{initial}{suffix}.pt"


def print_sweep(budget, initial, record):
    print(
        f"{budget} {initial} sweep {record.sweep} Q {record.q_lambda:.8e} "
        f"rel {record.relative_decrease:.3e} s_max {record.stationarity_max:.3e} "
        f"retry {any(record.retried)} monotone {record.monotone_ok} seconds {record.seconds:.2f}",
        flush=True,
    )


## 5. 試運転

- 入力: `JOBS`、`SMOKE_RULE`、`CONFIG`、ベンチマークの `cuda_float32` の 5 行。
- このステップで行うこと: `*_smoke.pt` だけを消してから、検算 1〜4、4 本の 5 掃引、再開の確認、時間の内訳と本実行の見積もりを行う。通らなければその場で止める。
- 行わないこと: 本実行のファイルの削除、規則や許容誤差の変更、`src` の修正。
- 出力: `checks_smoke.json`、`history_smoke.csv`、`diagnostics_smoke.csv`、`failures_smoke.csv` ではなく `failures_smoke.json`、`timing_smoke.json`。変数 `SMOKE_RESULTS`。
- 完了条件: 検算 1 の相対差が 1e-10 以下。検算 2 の相対差の最大が 1e-6 以下。再開は `resumed_from_sweep == 2` で、時間以外の記録とコアと診断がビット単位で一致。検算 3 は全掃引で `monotone_ok`。検算 4 は最終 Q_λ の相対差が 1e-10 以下。
- 次のステップへの接続: 6 以降は `RUN_FULL` が真のときだけこれらの関数を本実行に使う。


In [10]:
from nn_compression.tt_mps.metric_tt_solver import run_metric_tt_als_until_converged
from nn_compression.utils.artifacts import atomic_write_csv

SMOKE_RESULTS = {}
CHECKS = {}


def run_smoke():
    if not RUN_SMOKE:
        print("試運転はしません。", flush=True)
        return
    for path in MODELS.glob("*_smoke.pt"):
        path.unlink()
        print("deleted", path.name, flush=True)

    def run_job(job, rule, path, metadata, progress):
        return run_metric_tt_als_until_converged(
            job["cores"],
            WEIGHT,
            G_LAMBDA,
            job["out_modes"],
            job["in_modes"],
            CONFIG,
            rule,
            report_gram=G,
            checkpoint_path=path,
            metadata=metadata,
            progress=progress,
            diagnostics=diagnostics_for_job(job),
        )

    for job in JOBS:
        if not (job["budget"] == "50" and job["initial"] == "M3"):
            continue
        metadata = job_metadata(job["budget"], job["initial"], "smoke")
        print("試運転", job["budget"], job["initial"], flush=True)
        result = run_job(
            job,
            SMOKE_RULE,
            checkpoint_path(job["budget"], job["initial"], "_smoke"),
            metadata,
            lambda record, job=job: print_sweep(job["budget"], job["initial"], record),
        )
        gap = relative_gap(result.initial_q_lambda, job["expected_q"])
        CHECKS["check1_first"] = {
            "budget": job["budget"],
            "initial": job["initial"],
            "initial_q_lambda": result.initial_q_lambda,
            "expected": job["expected_q"],
            "relative_gap": gap,
            "passed": gap <= 1e-10,
        }
        print("検算 1", job["budget"], job["initial"], f"{gap:.3e}", flush=True)
        if gap > 1e-10:
            atomic_write_json(RESULTS / "checks_smoke.json", CHECKS)
            raise RuntimeError(f"検算 1 が通りません: 相対差 {gap}")
        reference = [
            float(row["q_lambda"])
            for row in REFERENCE["als"]
            if row["tensorization"] == "T2" and row["budget"] == "50" and row["solver"] == "cuda_float32"
        ]
        if len(reference) != 5 or len(result.history) != 5:
            raise RuntimeError(f"検算 2 の行数が参照 {len(reference)}、今回 {len(result.history)} です。")
        gaps = [relative_gap(record.q_lambda, expected) for record, expected in zip(result.history, reference)]
        bitwise = all(record.q_lambda == expected for record, expected in zip(result.history, reference))
        CHECKS["check2"] = {
            "q_lambda": [record.q_lambda for record in result.history],
            "reference": reference,
            "relative_gaps": gaps,
            "max_relative_gap": max(gaps),
            "bitwise_equal": bitwise,
            "passed": max(gaps) <= 1e-6,
        }
        print("検算 2", "max_rel", f"{max(gaps):.3e}", "bitwise", bitwise, flush=True)
        if max(gaps) > 1e-6:
            atomic_write_json(RESULTS / "checks_smoke.json", CHECKS)
            raise RuntimeError(f"検算 2 が通りません: 最大相対差 {max(gaps)}")
        SMOKE_RESULTS[(job["budget"], job["initial"])] = result

    for job in JOBS:
        key = (job["budget"], job["initial"])
        if key in SMOKE_RESULTS:
            continue
        metadata = job_metadata(job["budget"], job["initial"], "smoke")
        print("試運転", job["budget"], job["initial"], flush=True)
        result = run_job(
            job,
            SMOKE_RULE,
            checkpoint_path(job["budget"], job["initial"], "_smoke"),
            metadata,
            lambda record, job=job: print_sweep(job["budget"], job["initial"], record),
        )
        gap = relative_gap(result.initial_q_lambda, job["expected_q"])
        print("検算 1", job["budget"], job["initial"], f"{gap:.3e}", flush=True)
        if gap > 1e-10:
            CHECKS.setdefault("check1", {})[f"{job['budget']}_{job['initial']}"] = gap
            atomic_write_json(RESULTS / "checks_smoke.json", CHECKS)
            raise RuntimeError(f"検算 1 が通りません: {job['budget']} {job['initial']} 相対差 {gap}")
        SMOKE_RESULTS[key] = result

    check1 = {}
    for job in JOBS:
        result = SMOKE_RESULTS[(job["budget"], job["initial"])]
        gap = relative_gap(result.initial_q_lambda, job["expected_q"])
        check1[f"{job['budget']}_{job['initial']}"] = {
            "initial_q_lambda": result.initial_q_lambda,
            "expected": job["expected_q"],
            "relative_gap": gap,
        }
    CHECKS["check1"] = {"runs": check1, "passed": all(item["relative_gap"] <= 1e-10 for item in check1.values())}

    timing = timing_report(SMOKE_RESULTS)
    atomic_write_json(RESULTS / "timing_smoke.json", timing)

    first = next(job for job in JOBS if job["budget"] == "50" and job["initial"] == "M3")
    uninterrupted = SMOKE_RESULTS[("50", "M3")]
    resume_path = MODELS / "long_als_T2_50_M3_resume_smoke.pt"
    metadata = job_metadata("50", "M3", "smoke")

    class _SmokeInterrupt(RuntimeError):
        pass

    def stop_at_three(record):
        print_sweep("50", "M3_resume_interrupt", record)
        if record.sweep == 3:
            raise _SmokeInterrupt

    try:
        run_job(first, SMOKE_RULE, resume_path, metadata, stop_at_three)
    except _SmokeInterrupt:
        print("3 掃引目で中断しました。", flush=True)
    else:
        raise RuntimeError("再開の確認で、3 掃引目の中断が起きませんでした。")
    resumed = run_job(
        first,
        SMOKE_RULE,
        resume_path,
        metadata,
        lambda record: print_sweep("50", "M3_resume", record),
    )
    differences = compare_runs(uninterrupted, resumed)
    CHECKS["resume"] = differences
    print("再開", "resumed_from_sweep", resumed.resumed_from_sweep, "match", differences["matched"], flush=True)
    atomic_write_json(RESULTS / "checks_smoke.json", CHECKS)
    print("本実行の見積もり 時間", f"{timing['estimate_hours_total']:.2f}", flush=True)
    if resumed.resumed_from_sweep != 2 or not differences["matched"]:
        raise RuntimeError(f"再開の確認が一致しません: {differences}")
    if timing["estimate_exceeds_5_hours"]:
        print("見積もりが 5 時間を超えます。本実行の前に確認が必要です。", flush=True)

    check3 = {}
    check4 = {}
    for job in JOBS:
        result = SMOKE_RESULTS[(job["budget"], job["initial"])]
        if not result.history:
            check3[f"{job['budget']}_{job['initial']}"] = "完了した掃引なし"
            check4[f"{job['budget']}_{job['initial']}"] = "完了した掃引なし"
            continue
        check3[f"{job['budget']}_{job['initial']}"] = all(record.monotone_ok for record in result.history)
        recomputed = float(q_lambda_of_cores(
            result.cores, WEIGHT, G_LAMBDA, job["out_modes"], job["in_modes"],
        ).item())
        recorded = result.history[-1].q_lambda
        check4[f"{job['budget']}_{job['initial']}"] = {
            "recomputed": recomputed,
            "recorded": recorded,
            "relative_gap": relative_gap(recomputed, recorded),
        }
    check3_passed = all(value is True for value in check3.values())
    check4_passed = all(
        isinstance(value, dict) and value["relative_gap"] <= 1e-10 for value in check4.values()
    )
    CHECKS["check3"] = {"runs": check3, "passed": check3_passed}
    CHECKS["check4"] = {"runs": check4, "passed": check4_passed}
    write_smoke_tables(SMOKE_RESULTS)
    atomic_write_json(RESULTS / "checks_smoke.json", CHECKS)
    if not check3_passed:
        raise RuntimeError(f"検算 3 が通りません: {check3}")
    if not check4_passed:
        raise RuntimeError(f"検算 4 が通りません: {check4}")
    print("試運転の検算 1〜4 と再開は通りました。", flush=True)


def compare_runs(uninterrupted, resumed):
    mismatched = []
    for index, (left, right) in enumerate(zip(uninterrupted.history, resumed.history, strict=False)):
        left_payload = {key: value for key, value in asdict(left).items() if key not in TIMING_FIELDS}
        right_payload = {key: value for key, value in asdict(right).items() if key not in TIMING_FIELDS}
        for key, value in left_payload.items():
            if value != right_payload[key]:
                mismatched.append(f"history[{index}].{key}")
    if len(uninterrupted.history) != len(resumed.history):
        mismatched.append("history_length")
    if len(uninterrupted.diagnostics) != len(resumed.diagnostics):
        mismatched.append("diagnostics_length")
    for index, (left, right) in enumerate(zip(uninterrupted.diagnostics, resumed.diagnostics)):
        for key in set(left) | set(right):
            if left.get(key) != right.get(key):
                mismatched.append(f"diagnostics[{index}].{key}")
    if uninterrupted.initial_q_lambda != resumed.initial_q_lambda:
        mismatched.append("initial_q_lambda")
    if uninterrupted.initial_q_unregularized != resumed.initial_q_unregularized:
        mismatched.append("initial_q_unregularized")
    if uninterrupted.stop_reason != resumed.stop_reason:
        mismatched.append("stop_reason")
    if uninterrupted.failure != resumed.failure:
        mismatched.append("failure")
    core_abs = 0.0
    core_rel = 0.0
    bitwise_cores = True
    for left, right in zip(uninterrupted.cores, resumed.cores):
        if not torch.equal(left, right):
            bitwise_cores = False
        gap = (left - right).abs()
        core_abs = max(core_abs, float(gap.max().item()))
        scale = float(left.abs().max().item())
        core_rel = max(core_rel, float(gap.max().item()) / scale if scale else float(gap.max().item()))
    if not bitwise_cores:
        mismatched.append("cores")
    return {
        "resumed_from_sweep": resumed.resumed_from_sweep,
        "matched": not mismatched and bitwise_cores,
        "mismatched_fields": mismatched,
        "core_max_abs": core_abs,
        "core_max_rel": core_rel,
    }


def timing_report(results):
    runs = {}
    estimate = 0.0
    for (budget, initial), result in results.items():
        body = []
        stationarity = []
        unregularized = []
        for record in result.history:
            body.append(record.seconds - record.seconds_stationarity - record.seconds_q_unregularized)
            stationarity.append(record.seconds_stationarity)
            unregularized.append(record.seconds_q_unregularized)
        later = [record.seconds for record in result.history if record.sweep >= 2]
        diag_times = []
        for item in result.save_timings:
            diag_times.append(float(item["seconds_diagnostics"]) + float(item["seconds_save"] or 0.0))
        mean_sweep = sum(later) / len(later)
        mean_diag = sum(diag_times) / len(diag_times)
        run_estimate = mean_sweep * 1000.0 + mean_diag * 20.0
        estimate += run_estimate
        runs[f"{budget}_{initial}"] = {
            "seconds_als_body": body,
            "seconds_stationarity": stationarity,
            "seconds_q_unregularized": unregularized,
            "seconds_diagnostics_and_save": diag_times,
            "mean_seconds_sweeps_2_to_5": mean_sweep,
            "mean_seconds_diagnostics_and_save": mean_diag,
            "estimate_seconds": run_estimate,
            "s_max": [record.stationarity_max for record in result.history],
            "q_lambda": [record.q_lambda for record in result.history],
            "any_retry": any(any(record.retried) for record in result.history),
            "failure": None if result.failure is None else asdict(result.failure),
            "monotone_ok": [record.monotone_ok for record in result.history],
        }
    return {
        "runs": runs,
        "estimate_seconds_total": estimate,
        "estimate_hours_total": estimate / 3600.0,
        "estimate_exceeds_5_hours": estimate > 5.0 * 3600.0,
        "estimate_definition": "掃引2〜5の seconds の平均×1000 + 診断と保存の1回あたりの平均×20。4本の和。",
    }


def write_smoke_tables(results):
    history = []
    diagnostics = []
    failures = []
    for job in JOBS:
        result = results[(job["budget"], job["initial"])]
        for record in result.history:
            history.append(history_row(job["budget"], job["initial"], record))
        for values, timing in zip(result.diagnostics, result.save_timings, strict=True):
            diagnostics.append(diagnostics_row(job["budget"], job["initial"], values, timing))
        if result.failure is not None:
            failures.append({"budget": job["budget"], "initial": job["initial"], **asdict(result.failure)})
    atomic_write_csv(
        RESULTS / "history_smoke.csv",
        history,
        ["budget", "initial", *HISTORY_FIELDS],
    )
    atomic_write_csv(
        RESULTS / "diagnostics_smoke.csv",
        diagnostics,
        ["budget", "initial", "sweep", *DIAGNOSTIC_KEYS, "seconds_diagnostics", "seconds_save"],
    )
    atomic_write_json(RESULTS / "failures_smoke.json", failures)


run_smoke()


試運転はしません。


## 6. 本実行

- 入力: `JOBS`、`FULL_RULE`、`CONFIG`。順序は 50%・M3、50%・M4_retruncated、25%・M3、25%・M4_retruncated。
- このステップで行うこと: `RUN_FULL` のときだけ 4 本を回す。保存ファイルがあればソルバーが続きから回す。10 掃引ごとに掃引、Q_λ、相対減少、s_max、解き直し、経過時間を出す。
- 行わないこと: `RUN_FULL` が偽のときの実行。試運転ファイルの削除。停止規則の変更。
- 出力: `models/long_als_T2_{budget}_{initial}.pt`。変数 `FULL_RESULTS`。
- 完了条件: 4 本の `FULL_RESULTS` がある。または `RUN_FULL` が偽で、このセルが何も走らせずに終わる。
- 次のステップへの接続: 7 は `FULL_RESULTS` の最終コアと履歴を検算する。


In [12]:
FULL_RESULTS = {}
FULL_ORDER = (("50", "M3"), ("50", "M4_retruncated"), ("25", "M3"), ("25", "M4_retruncated"))


def run_full():
    if not RUN_FULL:
        print("本実行はしません。", flush=True)
        return
    atomic_write_json(RESULTS / "settings.json", SETTINGS)
    for budget, initial in FULL_ORDER:
        job = next(item for item in JOBS if item["budget"] == budget and item["initial"] == initial)
        started = time.perf_counter()

        def progress(record, budget=budget, initial=initial, started=started):
            if record.sweep % 10 == 0 or record.stop_reason is not None:
                print(
                    f"{budget} {initial} sweep {record.sweep} Q {record.q_lambda:.8e} "
                    f"rel {record.relative_decrease:.3e} s_max {record.stationarity_max:.3e} "
                    f"retry {any(record.retried)} elapsed {time.perf_counter() - started:.1f}",
                    flush=True,
                )

        print("本実行", budget, initial, flush=True)
        FULL_RESULTS[(budget, initial)] = run_metric_tt_als_until_converged(
            job["cores"],
            WEIGHT,
            G_LAMBDA,
            job["out_modes"],
            job["in_modes"],
            CONFIG,
            FULL_RULE,
            report_gram=G,
            checkpoint_path=checkpoint_path(budget, initial, ""),
            metadata=job_metadata(budget, initial, "full"),
            progress=progress,
            diagnostics=diagnostics_for_job(job),
        )
        print("停止", budget, initial, FULL_RESULTS[(budget, initial)].stop_reason, flush=True)


run_full()


本実行 50 M3
50 M3 sweep 10 Q 3.00185887e+07 rel 4.494e-03 s_max 8.234e-03 retry False elapsed 48.9
50 M3 sweep 20 Q 2.93729796e+07 rel 1.208e-03 s_max 4.991e-03 retry False elapsed 95.3
50 M3 sweep 30 Q 2.91424512e+07 rel 5.554e-04 s_max 3.718e-03 retry False elapsed 147.7
50 M3 sweep 40 Q 2.90243531e+07 rel 3.139e-04 s_max 2.962e-03 retry False elapsed 281.1
50 M3 sweep 50 Q 2.89528887e+07 rel 2.030e-04 s_max 2.472e-03 retry False elapsed 397.4
50 M3 sweep 60 Q 2.89041433e+07 rel 1.452e-04 s_max 2.141e-03 retry False elapsed 563.1
50 M3 sweep 70 Q 2.88680367e+07 rel 1.107e-04 s_max 1.902e-03 retry False elapsed 711.2
50 M3 sweep 80 Q 2.88400053e+07 rel 8.722e-05 s_max 1.712e-03 retry False elapsed 850.4
50 M3 sweep 90 Q 2.88177189e+07 rel 6.992e-05 s_max 1.550e-03 retry False elapsed 902.2
50 M3 sweep 100 Q 2.87997401e+07 rel 5.679e-05 s_max 1.408e-03 retry False elapsed 1033.9
50 M3 sweep 110 Q 2.87850459e+07 rel 4.673e-05 s_max 1.284e-03 retry False elapsed 1187.4
50 M3 sweep 120 Q 2.

## 7. 本実行の検算 3・4

- 入力: `FULL_RESULTS`。
- このステップで行うこと: `RUN_FULL` のとき、4 本について単調性と、最終コアから計算し直した Q_λ を記録する。通らなければ集計に進まず止める。
- 行わないこと: `RUN_FULL` が偽のときの検算。規則の変更。
- 出力: `FULL_CHECKS`。
- 完了条件: 4 本とも `monotone_ok` が真で、Q_λ の相対差が 1e-10 以下。履歴が空の本は「完了した掃引なし」。
- 次のステップへの接続: 8 は検算を通った 4 本を予算ごとに比べる。


In [14]:
FULL_CHECKS = {}


def check_full():
    if not RUN_FULL:
        print("本実行の検算はしません。", flush=True)
        return
    check3 = {}
    check4 = {}
    for budget, initial in FULL_ORDER:
        job = next(item for item in JOBS if item["budget"] == budget and item["initial"] == initial)
        result = FULL_RESULTS[(budget, initial)]
        name = f"{budget}_{initial}"
        if not result.history:
            check3[name] = "完了した掃引なし"
            check4[name] = "完了した掃引なし"
            continue
        check3[name] = all(record.monotone_ok for record in result.history)
        recomputed = float(q_lambda_of_cores(
            result.cores, WEIGHT, G_LAMBDA, job["out_modes"], job["in_modes"],
        ).item())
        recorded = result.history[-1].q_lambda
        check4[name] = {
            "recomputed": recomputed,
            "recorded": recorded,
            "relative_gap": relative_gap(recomputed, recorded),
        }
    passed = (
        all(value is True for value in check3.values())
        and all(isinstance(value, dict) and value["relative_gap"] <= 1e-10 for value in check4.values())
    )
    FULL_CHECKS["check3"] = check3
    FULL_CHECKS["check4"] = check4
    FULL_CHECKS["passed"] = passed
    if not passed:
        raise RuntimeError(f"本実行の検算 3・4 が通りません: {FULL_CHECKS}")
    print("本実行の検算 3・4 は通りました。", flush=True)


check_full()


本実行の検算 3・4 は通りました。


## 8. 集計と判定

- 入力: `FULL_RESULTS`、予備実験の `m5_history.csv` と `rho.csv`。等価幅は `EQUIVALENCE_TOLERANCE`。
- このステップで行うこと: 予算ごとに δ_Q、5 種類の相対距離、0.1% 以内に入った掃引、50 掃引目との比較、ρ、判定を作る。結果を見て規則や幅は変えない。
- 行わないこと: 本実行が無いときの計算。ρ の区間。検証データを停止に使うこと。
- 出力: `comparison.csv`、`rho.csv`、`decision.json`。変数 `DECISIONS`。
- 完了条件: 予算 50 と 25 の両方に判定がある。
- 次のステップへの接続: 9 は `converged_initialization_dependent` の予算だけ摂動の 1 本を足す。


In [16]:
from nn_compression.metrics.als_convergence import classify_als_initialization_dependence

DECISIONS = {}
COMPARISON_ROWS = []
RHO_ROWS = []


def pilot_q_at_sweep_50(budget, initial):
    matches = [
        row for row in REFERENCE["history"]
        if row["calibration_set"] == "full"
        and row["tensorization"] == "T2"
        and row["budget"] == budget
        and row["initial"] == initial
        and int(row["sweep"]) == 50
    ]
    if len(matches) != 1:
        raise RuntimeError(f"50 掃引目の参照が {len(matches)} 行です: {budget} {initial}")
    return float(matches[0]["q_lambda"])


def pilot_reference_e(budget, domain):
    matches = [
        row for row in REFERENCE["rho"]
        if row["calibration_set"] == "full"
        and row["tensorization"] == "T2"
        and row["budget"] == budget
        and row["domain"] == domain
    ]
    if len(matches) != 1:
        raise RuntimeError(f"ρ の参照が {len(matches)} 行です: {budget} {domain}")
    return float(matches[0]["e_m2"]), float(matches[0]["e_m3"])


def first_sweep_within(history, tolerance):
    final = history[-1].q_lambda
    for record in history:
        if record.q_lambda <= final * (1.0 + tolerance):
            return record.sweep
    return None


def gram_distance(left, right, gram):
    numerator = squared_output_error_from_gram(left - right, gram)
    denominator = squared_output_error_from_gram(WEIGHT, gram)
    return float(torch.sqrt(numerator / denominator).item())


def aggregate():
    if not RUN_FULL or not FULL_RESULTS:
        print("集計は、本実行の結果があるときだけ行います。", flush=True)
        return
    for budget in ("50", "25"):
        job_m3 = next(item for item in JOBS if item["budget"] == budget and item["initial"] == "M3")
        result_m3 = FULL_RESULTS[(budget, "M3")]
        result_m4 = FULL_RESULTS[(budget, "M4_retruncated")]
        if not result_m3.history or not result_m4.history:
            raise RuntimeError(f"{budget} は完了した掃引がないので、判定に使う最終の Q_λ がありません。")
        decision = classify_als_initialization_dependence(
            result_m3.stop_reason,
            result_m4.stop_reason,
            result_m3.history[-1].q_lambda,
            result_m4.history[-1].q_lambda,
            equivalence_tolerance=EQUIVALENCE_TOLERANCE,
        )
        DECISIONS[budget] = {
            "verdict": decision.verdict,
            "delta_q": decision.delta_q,
            "equivalence_tolerance": decision.equivalence_tolerance,
        }
        dense_m3 = tt_matrix_to_dense(result_m3.cores, job_m3["out_modes"], job_m3["in_modes"])
        dense_m4 = tt_matrix_to_dense(result_m4.cores, job_m3["out_modes"], job_m3["in_modes"])
        frobenius = float(
            (torch.linalg.matrix_norm(dense_m3 - dense_m4) / torch.linalg.matrix_norm(WEIGHT)).item()
        )
        distances = {
            "frobenius": frobenius,
            "g_lambda": gram_distance(dense_m3, dense_m4, G_LAMBDA),
            "g": gram_distance(dense_m3, dense_m4, G),
            "validation_same": gram_distance(dense_m3, dense_m4, VALIDATION["validation_same"]["gram"]),
            "validation_other": gram_distance(dense_m3, dense_m4, VALIDATION["validation_other"]["gram"]),
        }
        row = {
            "budget": budget,
            "stop_M3": result_m3.stop_reason,
            "stop_M4_retruncated": result_m4.stop_reason,
            "sweeps_M3": len(result_m3.history),
            "sweeps_M4_retruncated": len(result_m4.history),
            "q_M3": result_m3.history[-1].q_lambda,
            "q_M4_retruncated": result_m4.history[-1].q_lambda,
            "delta_q": decision.delta_q,
            "verdict": decision.verdict,
            "sweep_within_0_1_percent_M3": first_sweep_within(result_m3.history, EQUIVALENCE_TOLERANCE),
            "sweep_within_0_1_percent_M4_retruncated": first_sweep_within(result_m4.history, EQUIVALENCE_TOLERANCE),
        }
        row.update(distances)
        for initial, result in (("M3", result_m3), ("M4_retruncated", result_m4)):
            pilot = pilot_q_at_sweep_50(budget, initial)
            ours = next((record.q_lambda for record in result.history if record.sweep == 50), None)
            row[f"q_pilot_sweep50_{initial}"] = pilot
            row[f"q_sweep50_{initial}"] = ours
            row[f"relative_sweep50_{initial}"] = None if ours is None else relative_gap(ours, pilot)
        COMPARISON_ROWS.append(row)
        for initial, result in (("M3", result_m3), ("M4_retruncated", result_m4)):
            by_sweep = {item["sweep"]: item for item in result.diagnostics}
            points = []
            if 50 in by_sweep:
                points.append(("50", by_sweep[50]))
            if result.diagnostics:
                points.append(("final", result.diagnostics[-1]))
            for point, values in points:
                for domain, e_key, gated_key in (
                    ("calibration", "e_calibration", None),
                    ("validation_same", "e_validation_same", "E_gated_validation_same"),
                    ("validation_other", "e_validation_other", "E_gated_validation_other"),
                ):
                    e_m2, e_m3 = pilot_reference_e(budget, domain)
                    e_m5 = values[e_key]
                    RHO_ROWS.append({
                        "budget": budget,
                        "initial": initial,
                        "domain": domain,
                        "point": point,
                        "e_m5": e_m5,
                        "e_m2": e_m2,
                        "e_m3": e_m3,
                        "rho": (e_m3 - e_m5) / (e_m3 - e_m2),
                        "E_gated": None if gated_key is None else values[gated_key],
                    })
    atomic_write_csv(RESULTS / "comparison.csv", COMPARISON_ROWS, list(COMPARISON_ROWS[0]))
    atomic_write_csv(RESULTS / "rho.csv", RHO_ROWS, list(RHO_ROWS[0]))
    atomic_write_json(RESULTS / "decision.json", {
        "equivalence_tolerance": EQUIVALENCE_TOLERANCE,
        "budgets": DECISIONS,
    })
    print(DECISIONS, flush=True)


aggregate()


{'50': {'verdict': 'local_solver_unresolved', 'delta_q': 0.007018345621307332, 'equivalence_tolerance': 0.001}, '25': {'verdict': 'max_sweeps_unresolved', 'delta_q': 0.0036021858921718446, 'equivalence_tolerance': 0.001}}


## 9. 摂動

- 入力: `DECISIONS` と、保存されたままの M3 コア。
- このステップで行うこと: 判定が `converged_initialization_dependent` の予算だけ、`perturb_tt_matrix_site`（site 1、相対 1e-2、seed 0）から `FULL_RULE` で 1 本回す。M3 との組にも判定関数を当てはめて記録するが、§9 の判定は変えない。
- 行わないこと: 条件を満たさない予算の実行。`decision.json` の上書き。
- 出力: `perturbation.csv`。無ければ作らない。
- 完了条件: 対象の予算ごとに、δ_W と、M3・M4_retruncated との δ_Q と 5 種類の距離が記録されている。
- 次のステップへの接続: 10 は本実行の履歴と失敗を保存する。


In [18]:
from nn_compression.metrics.als_convergence import relative_objective_difference
from nn_compression.tt_mps.metric_tt_solver import perturb_tt_matrix_site

PERTURBATION_ROWS = []


def run_perturbations():
    if not RUN_FULL or not DECISIONS:
        print("摂動は、本実行の判定があるときだけ検討します。", flush=True)
        return
    targets = [budget for budget, item in DECISIONS.items() if item["verdict"] == "converged_initialization_dependent"]
    if not targets:
        print("摂動の対象になる予算はありません。", flush=True)
        return
    for budget in targets:
        job = next(item for item in JOBS if item["budget"] == budget and item["initial"] == "M3")
        perturbed, delta_w = perturb_tt_matrix_site(
            job["cores"], job["out_modes"], job["in_modes"], site=1, relative_size=1e-2, seed=0,
        )
        print(budget, "delta_w", f"{delta_w:.6e}", flush=True)
        started = time.perf_counter()

        def progress(record, budget=budget, started=started):
            if record.sweep % 10 == 0 or record.stop_reason is not None:
                print(
                    f"{budget} M3_perturbed_site1 sweep {record.sweep} Q {record.q_lambda:.8e} "
                    f"s_max {record.stationarity_max:.3e} elapsed {time.perf_counter() - started:.1f}",
                    flush=True,
                )

        result = run_metric_tt_als_until_converged(
            perturbed,
            WEIGHT,
            G_LAMBDA,
            job["out_modes"],
            job["in_modes"],
            CONFIG,
            FULL_RULE,
            report_gram=G,
            checkpoint_path=checkpoint_path(budget, "M3_perturbed_site1", ""),
            metadata=job_metadata(budget, "M3_perturbed_site1", "full"),
            progress=progress,
            diagnostics=diagnostics_for_job(job),
        )
        reference = classify_als_initialization_dependence(
            FULL_RESULTS[(budget, "M3")].stop_reason,
            result.stop_reason,
            FULL_RESULTS[(budget, "M3")].history[-1].q_lambda,
            result.history[-1].q_lambda,
            equivalence_tolerance=EQUIVALENCE_TOLERANCE,
        )
        dense = tt_matrix_to_dense(result.cores, job["out_modes"], job["in_modes"])
        row = {
            "budget": budget,
            "delta_w": delta_w,
            "stop_reason": result.stop_reason,
            "q_lambda": result.history[-1].q_lambda,
            "delta_q_vs_M3": relative_objective_difference(
                result.history[-1].q_lambda, FULL_RESULTS[(budget, "M3")].history[-1].q_lambda,
            ),
            "delta_q_vs_M4_retruncated": relative_objective_difference(
                result.history[-1].q_lambda, FULL_RESULTS[(budget, "M4_retruncated")].history[-1].q_lambda,
            ),
            "reference_verdict_vs_M3": reference.verdict,
        }
        for initial in ("M3", "M4_retruncated"):
            other = tt_matrix_to_dense(
                FULL_RESULTS[(budget, initial)].cores, job["out_modes"], job["in_modes"],
            )
            prefix = "M3" if initial == "M3" else "M4_retruncated"
            row[f"frobenius_vs_{prefix}"] = float(
                (torch.linalg.matrix_norm(dense - other) / torch.linalg.matrix_norm(WEIGHT)).item()
            )
            row[f"g_lambda_vs_{prefix}"] = gram_distance(dense, other, G_LAMBDA)
            row[f"g_vs_{prefix}"] = gram_distance(dense, other, G)
            row[f"validation_same_vs_{prefix}"] = gram_distance(dense, other, VALIDATION["validation_same"]["gram"])
            row[f"validation_other_vs_{prefix}"] = gram_distance(dense, other, VALIDATION["validation_other"]["gram"])
        PERTURBATION_ROWS.append(row)
    atomic_write_csv(RESULTS / "perturbation.csv", PERTURBATION_ROWS, list(PERTURBATION_ROWS[0]))


run_perturbations()


摂動の対象になる予算はありません。


## 10. 本実行の保存

- 入力: `FULL_RESULTS`。
- このステップで行うこと: 本実行が終わっているとき、履歴・診断・失敗を results に書く。試運転の `_smoke` ファイルはここでは触らない。
- 行わないこと: 結果が無いときの空ファイルの作成。比較表の再計算。
- 出力: `history.csv`、`diagnostics.csv`、`failures.json`。
- 完了条件: 行数が、4 本の掃引数および保存回数と一致する。
- 次のステップへの接続: なし。


In [20]:
def save_full():
    if not RUN_FULL or not FULL_RESULTS:
        print("本実行の保存はしません。", flush=True)
        return
    history = []
    diagnostics = []
    failures = []
    for budget, initial in FULL_ORDER:
        result = FULL_RESULTS[(budget, initial)]
        for record in result.history:
            history.append(history_row(budget, initial, record))
        for values, timing in zip(result.diagnostics, result.save_timings, strict=True):
            diagnostics.append(diagnostics_row(budget, initial, values, timing))
        if result.failure is not None:
            failures.append({"budget": budget, "initial": initial, **asdict(result.failure)})
    atomic_write_csv(RESULTS / "history.csv", history, ["budget", "initial", *HISTORY_FIELDS])
    atomic_write_csv(
        RESULTS / "diagnostics.csv",
        diagnostics,
        ["budget", "initial", "sweep", *DIAGNOSTIC_KEYS, "seconds_diagnostics", "seconds_save"],
    )
    atomic_write_json(RESULTS / "failures.json", failures)
    if FULL_CHECKS:
        atomic_write_json(RESULTS / "checks.json", FULL_CHECKS)
    print("history", len(history), "diagnostics", len(diagnostics), "failures", len(failures), flush=True)


save_full()


history 3426 diagnostics 69 failures 1
